In [ ]:
# Install the latest ultralytics for YOLO26 support
!pip install -q ultralytics roboflow
import os
import torch
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 75.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo se

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="#######################") #upload dataset to roboflow and get your own api key
project = rf.workspace("mohameds-workspace-ikdw7").project("helmet_detection-hema-gretv")
version = project.version(1)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Helmet_detection-hema-1 in yolo26:: 100%|██████████| 11702/11702 [00:01<00:00, 10890.86it/s]


In [ ]:
# Load the YOLO26-S model
model = YOLO('yolo26s.pt')

# Training Configuration
results = model.train(
    data="/kaggle/working/Helmet_detection-hema-1/data.yaml",
    epochs=150,           # As discussed, 150 is the sweet spot
    imgsz=640,            # 640 is standard, 800 for very small helmets
    batch=64,             # Total batch size (32 per GPU)
    device=[0, 1],        # Trigger Multi-GPU (DDP)
    optimizer='MuSGD',    # The stable 2026 optimizer
    patience=50,          # Early stopping
    save=True,
    project='PetroHSE_Safety',
    name='Helmet_Color_Detection',
    exist_ok=True
)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Helmet_detection-hema-1/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

In [ ]:
# Load the best trained model
best_model = YOLO('/kaggle/working/PetroHSE_Safety/Helmet_Color_Detection/weights/best.pt')

# Run validation
metrics = best_model.val()
print(f"Final mAP 50-95: {metrics.box.map}")

# Run a test prediction on a single image
results = best_model.predict(source=f"{dataset.location}/test/images", save=True, conf=0.5)

# Display one result
import matplotlib.pyplot as plt
import cv2

# Get path to the first prediction result
res_path = os.path.join(results[0].save_dir, os.listdir(results[0].save_dir)[0])
img = cv2.cvtColor(cv2.imread(res_path), cv2.COLOR_BGR2RGB)
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Export to ONNX (Best for general CPU/Streamlit usage)
best_model.export(format='onnx', imgsz=640)

# Export to TensorRT (Best if the site uses NVIDIA Jetson or Tesla GPUs)
# best_model.export(format='engine', device=0)